# Three small data modalities

Text, tabular data, and recommendations look different at the input boundary, but the modeling questions are similar: what is the representation, what is the baseline, and where can the model fail? Each section below is intentionally compact and independent.

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import torch
from torch import nn

sns.set_theme(style='whitegrid', context='notebook')
torch.manual_seed(2); np.random.seed(2)

## Text: characters become integer ids

A tokenizer is a reversible mapping between symbols and integers. A language model receives a context and predicts a distribution over the next symbol. The important distinction is between the discrete id used for lookup and the learned continuous embedding used by the network.

In [ ]:
text = 'good models explain their mistakes'
vocabulary = sorted(set(text))
char_to_id = {ch: i for i, ch in enumerate(vocabulary)}
id_to_char = {i: ch for ch, i in char_to_id.items()}
encoded = torch.tensor([char_to_id[ch] for ch in text])
print('vocabulary:', ''.join(vocabulary))
print('round trip:', ''.join(id_to_char[i.item()] for i in encoded))
embedding = nn.Embedding(len(vocabulary), 8)
print('embedding shape for one context:', embedding(encoded[:5]).shape)

## Tabular data: categorical embeddings plus continuous features

A categorical column cannot be averaged as if its numeric codes were meaningful measurements. An embedding gives each category a learned vector. Continuous columns are normalized separately and concatenated with those vectors before the prediction head.

In [ ]:
rows = pd.DataFrame({
    'device': ['mobile','desktop','mobile','tablet','desktop','mobile'],
    'region': ['south','north','north','south','west','west'],
    'hours': [1.2, 4.1, 2.4, 3.0, 5.2, 0.8],
    'converted': [0, 1, 1, 0, 1, 0],
})
encoders = {column: {value: i for i, value in enumerate(sorted(rows[column].unique()))}
            for column in ['device','region']}
cat = torch.tensor([[encoders[c][v] for c, v in [('device', row.device), ('region', row.region)]]
                    for row in rows.itertuples()])
continuous = torch.tensor(rows[['hours']].to_numpy(), dtype=torch.float32)
embeddings = [nn.Embedding(len(encoders[c]), 3) for c in ['device','region']]
representation = torch.cat([embedding(cat[:, j]) for j, embedding in enumerate(embeddings)] + [continuous], dim=1)
print('categorical indices:', cat.tolist())
print('combined representation:', tuple(representation.shape))

## Collaborative filtering: learn users and items in the same space

A simple recommender represents each user and item by a vector. Their dot product estimates an affinity. The model learns from observed ratings, but the unobserved entries are not automatically negative examples. Cold-start users and items remain a limitation because they have no interaction history to identify their vectors.

In [ ]:
ratings = torch.tensor([[5., 0., 3., 0.], [0., 4., 0., 2.], [1., 0., 4., 0.]])
observed = ratings > 0
user_embedding = nn.Embedding(3, 2)
item_embedding = nn.Embedding(4, 2)
user_bias = nn.Embedding(3, 1)
item_bias = nn.Embedding(4, 1)
users, items = torch.where(observed)
predicted = (user_embedding(users) * item_embedding(items)).sum(1) + user_bias(users).squeeze(1) + item_bias(items).squeeze(1)
print('observed interactions:', len(users))
print('predictions for observed pairs:', predicted.detach().numpy().round(2))

The repository already contains a dedicated transformer sequence. This notebook therefore stops at representations and baselines instead of reproducing attention. Its purpose is to show how the input boundary differs across modalities and why a baseline remains necessary.